In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)


In [28]:
from pubsub.client import PubSubClient 

In [31]:
pubsub = PubSubClient(project_id="rmr-cloud-services")

In [ ]:
# pubsub.list_topics()

['container-analysis-notes-v1',
 'container-analysis-occurrences-v1beta1',
 'my-topic',
 'container-analysis-notes-v1beta1',
 'container-analysis-occurrences-v1',
 'filescom-events',
 'clickup-events',
 'filescom-dlq',
 'handoff-events']

['container-analysis-notes-v1',
 'container-analysis-occurrences-v1beta1',
 'my-topic',
 'container-analysis-notes-v1beta1',
 'container-analysis-occurrences-v1',
 'filescom-events',
 'clickup-events',
 'filescom-dlq',
 'handoff-events']

In [4]:
from pubsub.client import PubSubClient

client = PubSubClient(project_id="rmr-cloud-services")

received = client.pull("filescom-dlq-sub", max_messages=20)

for r in received:
    print(r.ack_id, r.message.data, r.message.attributes)

# leave them alone to view again later, or once you decide:
# client.ack("my-deadletter-sub", [r.ack_id for r in received])   # remove
# client.nack("my-deadletter-sub", [r.ack_id for r in received])  # put back now


RFAGFixdRkhRNxkIaFEOT14jPzUgKEUQAQgUBXx9cVpbdV9cGgdRDRlyfDB1aVwRAgEQW35VWxENem1cbdnTuvxEX0B3a18XBARBVXpWUhkFbV9fcC-QtdqozZ7Kf0AvOdzH5Jtpe_PDoK1uZiA9XxJLLD5-NDxFQV5AEkw_CURJUytDCypYEU4EISE-MD5FUw b'{"default":{"source":"Files.com"},"action":"destroy","interface":"desktop","path":"Clients/Orange Tree Co-RMROTREE/File Feeds/coffee garden.txt","at":"2026-08-24T19:36:15-04:00","username":"james.richmond@rmrbenefits.com","ip":"73.20.57.233","type":"file","size":33}' {'CloudPubSubDeadLetterSourceTopicPublishTime': '2026-08-24T23:36:17.271+00:00', 'CloudPubSubDeadLetterSourceSubscription': 'filescom-events-sub', 'CloudPubSubDeadLetterSourceDeliveryCount': '5', 'CloudPubSubDeadLetterSourceSubscriptionProject': 'rmr-cloud-services'}
RFAGFixdRkhRNxkIaFEOT14jPzUgKEUQAQgUBXx9cVpbdV9cGgdRDRlyfDB1aVwRAgEQW35VWhENem1cbdnTuvxEX0B3a1wQAAtAVXpeUxoMYVRcdC-QtdqozZ7Kf0AvOdzH5Jtpe_PDoK1uZiA9XxJLLD5-NDxFQV5AEkw_CURJUytDCypYEU4EISE-MD5FUw b'{"default":{"source":"Files.com"},"action":"move","interface":"desktop

In [6]:
from google.cloud import monitoring_v3
import time

def undelivered_count(project_id: str, subscription: str) -> int:
    client = monitoring_v3.MetricServiceClient()
    now = time.time()
    interval = monitoring_v3.TimeInterval(
        end_time={"seconds": int(now)},
        start_time={"seconds": int(now) - 300},
    )
    results = client.list_time_series(
        request={
            "name": f"projects/{project_id}",
            "filter": (
                'metric.type="pubsub.googleapis.com/subscription/num_undelivered_messages" '
                f'AND resource.labels.subscription_id="{subscription}"'
            ),
            "interval": interval,
            "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
        }
    )
    for series in results:
        return int(series.points[0].value.int64_value)
    return 0


In [11]:
count = undelivered_count(project_id="rmr-cloud-services", subscription="filescom-dlq-sub")
count

2763

2763